# Transform claims_and_billing

In [0]:
path = '/Volumes/workspace/default/ca_hospital_dataset/claims_and_billing.csv'

df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(path)
)
display(df)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

print(df.count())
null_counts = df.select([
    F.count(F.when(F.col(c).isNull(), 1)).alias(c) 
    for c in df.columns
])

df.printSchema()
display(null_counts)

In [0]:
display(df.select("insurance_provider").distinct())
display(df.select("payment_method").distinct())
display(df.select("claim_status").distinct())

In [0]:
display(df.filter(~(F.col("paid_amount") == 0) & (F.col("claim_status") == "Denied")))
display(df.filter((F.col("paid_amount") == 0) & ~(F.col("claim_status") == "Denied")))
display(df.filter((F.col("paid_amount") == 0) & (F.col("claim_status") == "Denied")))

In [0]:
display(df.filter(~(F.col("payment_method") == "Selfpay") & (F.col("claim_id").isNull())))
display(df.filter((F.col("payment_method") == "Selfpay") & ~(F.col("claim_id").isNull())))
display(df.filter((F.col("payment_method") == "Selfpay") & (F.col("claim_id").isNull())))

In [0]:
df = df.withColumn(
    "claim_billing_date", 
    F.to_date(F.to_timestamp(F.col("claim_billing_date"), "dd-MM-yyyy HH:mm"))
)

df.printSchema()
display(df)

In [0]:
table_name = "workspace.silver.claims_and_billing"

(df.write 
  .format("delta") 
  .mode("overwrite") 
  .saveAsTable(table_name)
  )

df = spark.read.table(table_name)

display(df)

# Transform patients

In [0]:
path = '/Volumes/workspace/default/ca_hospital_dataset/patients.csv'

df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(path)
)
display(df)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

print(df.count())
null_counts = df.select([
    F.count(F.when(F.col(c).isNull(), 1)).alias(c) 
    for c in df.columns
])

df.printSchema()
display(null_counts)

In [0]:
display(df.select("gender").distinct())
display(df.select("ethnicity").distinct())
display(df.select("insurance_type").distinct())
display(df.select("marital_status").distinct())
display(df.select("city").distinct())
display(df.select("state").distinct())

In [0]:
zip_code_length_check = df.select(
    F.length(F.col('zip').cast("string")).alias("zip_code_length")
).groupBy("zip_code_length").count().orderBy("zip_code_length")

display(zip_code_length_check)

In [0]:
df = df.withColumn(
    "zip", 
    F.col('zip').cast("string")
)

df.printSchema()
display(df)

In [0]:
table_name = "workspace.silver.patients"

(df.write 
  .format("delta") 
  .mode("overwrite") 
  .saveAsTable(table_name)
  )

df = spark.read.table(table_name)

display(df)

# Transform providers

In [0]:
path = '/Volumes/workspace/default/ca_hospital_dataset/providers.csv'

df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(path)
)
display(df)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

print(df.count())
null_counts = df.select([
    F.count(F.when(F.col(c).isNull(), 1)).alias(c) 
    for c in df.columns
])

df.printSchema()
display(null_counts)

In [0]:
display(df.select("department").distinct())
display(df.select("specialty").distinct())
display(df.select("inhouse").distinct())
display(df.select("location").distinct())

In [0]:
npi_length_check = df.select(
    F.length(F.col('npi').cast("string")).alias("npi_length")
).groupBy("npi_length").count().orderBy("npi_length")

display(npi_length_check)

In [0]:
df = df.withColumn(
    "npi", 
    F.col('npi').cast("string")
)

df.printSchema()
display(df)

In [0]:
table_name = "workspace.silver.providers"

(df.write 
  .format("delta") 
  .mode("overwrite") 
  .saveAsTable(table_name)
  )

df = spark.read.table(table_name)

display(df)

# Transform encounters

In [0]:
path = '/Volumes/workspace/default/ca_hospital_dataset/encounters.csv'

df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(path)
)
display(df)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

print(df.count())
null_counts = df.select([
    F.count(F.when(F.col(c).isNull(), 1)).alias(c) 
    for c in df.columns
])

df.printSchema()
display(null_counts)

In [0]:
display(df.select("visit_type").distinct())
display(df.select("department").distinct())
display(df.select("reason_for_visit").distinct())
display(df.select("diagnosis_code").distinct())
display(df.select("admission_type").distinct())
display(df.select("status").distinct())
display(df.select("readmitted_flag").distinct())

In [0]:
display(df.filter((F.col("visit_type") == "Outpatient") & ~(F.col("admission_type").isNull())))
display(df.filter((F.col("visit_type") == "Outpatient") & (F.col("admission_type").isNull())))

In [0]:
display(df.filter((F.col("visit_type") == "Telehealth") & ~(F.col("admission_type").isNull())))
display(df.filter((F.col("visit_type") == "Telehealth") & (F.col("admission_type").isNull())))

In [0]:
table_name = "workspace.silver.encounters"

(df.write 
  .format("delta") 
  .mode("overwrite") 
  .saveAsTable(table_name)
  )

df = spark.read.table(table_name)

display(df)